# פרויקט לימוד מכונה - חלק ב'

במחברת זו נבנה תהליך עבודה מסודר עבור חלק ב' של הפרויקט: טעינת הנתונים, הכנתם לאימון, חלוקה לסט אימון וסט אימות, ובהמשך אימון והשוואה בין מודלים שונים.

בשלב הנוכחי המחברת כוללת את שלד העבודה ואת שלב טעינת והכנת הנתונים בלבד. סעיפי המודלים מופיעים ככותרות להמשך, אך עדיין לא ממומשים.

## 1. ייבוא ספריות והגדרות ראשוניות

נייבא ספריות בסיסיות הדרושות לטעינת הנתונים, בדיקה ראשונית וחלוקה לסט אימון וסט אימות. בהמשך נוסיף ספריות נוספות רק כאשר נגיע לסעיפי המודלים.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

## 2. טעינת קבצי הנתונים

נטען את קובץ האימון, קובץ המבחן הסופי וקובץ הדוגמה להגשה. בשלב זה קובץ המבחן הסופי משמש רק לבדיקת מבנה, ולא לבחירת מודלים או לכוונון שלהם.

In [ ]:
BASE_DIR = Path.cwd()

TRAIN_FILE = BASE_DIR / "Xy_train.csv"
TEST_FILE = BASE_DIR / "X_test.csv"
EXAMPLE_SUBMISSION_FILE = BASE_DIR / "y test example.xlsx"

required_files = [TRAIN_FILE, TEST_FILE, EXAMPLE_SUBMISSION_FILE]
missing_files = [file_path.name for file_path in required_files if not file_path.exists()]

if missing_files:
    raise FileNotFoundError(f"הקבצים הבאים חסרים בתיקיית העבודה: {missing_files}")

train_raw = pd.read_csv(TRAIN_FILE)
test_raw = pd.read_csv(TEST_FILE)
example_submission = pd.read_excel(EXAMPLE_SUBMISSION_FILE)

display(train_raw.head())
display(test_raw.head())
display(example_submission.head())

## 3. בדיקות מבנה בסיסיות

נבדוק את גודל הקבצים, שמות העמודות, ערכים חסרים וכפילויות. בדיקות אלה נועדו לוודא שאנחנו עובדים עם הקבצים הנכונים לפני שמתחילים לבנות מודלים.

In [ ]:
data_summary = pd.DataFrame(
    {
        "קובץ": ["אימון", "מבחן סופי", "דוגמת הגשה"],
        "מספר שורות": [len(train_raw), len(test_raw), len(example_submission)],
        "מספר עמודות": [train_raw.shape[1], test_raw.shape[1], example_submission.shape[1]],
        "מספר ערכים חסרים": [
            int(train_raw.isna().sum().sum()),
            int(test_raw.isna().sum().sum()),
            int(example_submission.isna().sum().sum()),
        ],
        "מספר שורות כפולות": [
            int(train_raw.duplicated().sum()),
            int(test_raw.duplicated().sum()),
            int(example_submission.duplicated().sum()),
        ],
    }
)

display(data_summary)

In [ ]:
columns_summary = pd.DataFrame(
    {
        "עמודות בקובץ האימון": pd.Series(train_raw.columns),
        "עמודות בקובץ המבחן הסופי": pd.Series(test_raw.columns),
        "עמודות בקובץ הדוגמה": pd.Series(example_submission.columns),
    }
)

display(columns_summary)

In [ ]:
missing_summary = pd.DataFrame(
    {
        "חסרים באימון": train_raw.isna().sum(),
        "חסרים במבחן הסופי": test_raw.isna().sum(),
    }
).fillna(0).astype(int)

display(missing_summary)

## 4. זיהוי עמודת המטרה והפרדת מאפיינים

נזהה את עמודת המטרה מתוך קובץ האימון בפועל. לא נניח מראש את שם העמודה או את צורת הכתיבה שלה. לאחר הזיהוי נפריד בין מאפייני הקלט לבין משתנה המטרה.

In [ ]:
train_columns = list(train_raw.columns)
test_columns = list(test_raw.columns)

columns_only_in_train = [column for column in train_columns if column not in test_columns]

if len(columns_only_in_train) != 1:
    raise ValueError(
        "לא ניתן לזהות באופן חד-משמעי את עמודת המטרה. "
        f"עמודות שמופיעות רק באימון: {columns_only_in_train}"
    )

target_column = columns_only_in_train[0]
feature_columns = [column for column in train_columns if column != target_column]

missing_in_test = [column for column in feature_columns if column not in test_columns]
extra_in_test = [column for column in test_columns if column not in feature_columns]

if missing_in_test or extra_in_test:
    raise ValueError(
        "מבנה עמודות המאפיינים באימון ובמבחן הסופי אינו תואם. "
        f"חסרות במבחן: {missing_in_test}; עודפות במבחן: {extra_in_test}"
    )

X = train_raw[feature_columns].copy()
y = train_raw[target_column].copy()
X_test_final = test_raw[feature_columns].copy()

print(f"עמודת המטרה שזוהתה: {target_column}")
print(f"מספר מאפיינים: {len(feature_columns)}")

In [ ]:
target_distribution = (
    y.value_counts(dropna=False)
    .rename_axis("ערך המטרה")
    .reset_index(name="מספר רשומות")
)
target_distribution["אחוז"] = (target_distribution["מספר רשומות"] / len(y) * 100).round(2)

display(target_distribution)

## 5. הכנת נתונים ראשונית לאימון ולאימות

בשלב זה לא נבצע עדיין עיבוד מקדים מלא, משום שהעיבוד הסופי צריך להיות מותאם למודלים השונים. כאן נבצע רק בדיקות הכנה בסיסיות ונשמור עותקים נקיים לעבודה בהמשך.

חשוב: קובץ המבחן הסופי אינו משתתף באימון, בבחירת מודלים או בכוונון. הוא יישמר בצד עד לשלב החיזוי הסופי.

In [ ]:
if y.isna().any():
    raise ValueError("נמצאו ערכים חסרים בעמודת המטרה. יש לטפל בכך לפני חלוקת הנתונים.")

original_test_index = X_test_final.index.copy()
original_test_length = len(X_test_final)

X_model = X.copy()
y_model = y.copy()

print("הנתונים מוכנים לחלוקה ראשונית לאימון ולאימות.")

## 6. חלוקה לסט אימון וסט אימות

נחלק את קובץ האימון לסט אימון ולסט אימות. סט האימות יישמר בצד וישמש להערכת ביצועי המודלים על נתונים שלא שימשו לאימון. קובץ המבחן הסופי נשאר מחוץ לתהליך זה.

In [ ]:
stratify_target = y_model if y_model.nunique(dropna=False) > 1 else None

X_train, X_valid, y_train, y_valid = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_target,
)

split_summary = pd.DataFrame(
    {
        "סט": ["אימון", "אימות", "מבחן סופי"],
        "מספר רשומות": [len(X_train), len(X_valid), len(X_test_final)],
        "מספר מאפיינים": [X_train.shape[1], X_valid.shape[1], X_test_final.shape[1]],
    }
)

display(split_summary)

In [ ]:
train_target_distribution = y_train.value_counts(normalize=True, dropna=False).rename("אימון")
valid_target_distribution = y_valid.value_counts(normalize=True, dropna=False).rename("אימות")

split_target_distribution = (
    pd.concat([train_target_distribution, valid_target_distribution], axis=1)
    .fillna(0)
    .mul(100)
    .round(2)
)

display(split_target_distribution)

In [ ]:
assert len(X_test_final) == original_test_length
assert X_test_final.index.equals(original_test_index)

print("בדיקת קובץ המבחן הסופי הסתיימה: מספר השורות והסדר המקורי נשמרו.")

## 7. עצי החלטה

בסעיף זה נבנה בהמשך עץ החלטה מלא, נכוונן היפר-פרמטרים, נציג את העץ הנבחר, ננתח חשיבות משתנים ונעביר רשומת אימות לדוגמה דרך העץ.

## 8. רשתות נוירונים / MLP

בסעיף זה נבנה בהמשך רשת נוירונים, נריץ מודל ברירת מחדל, נכוונן היפר-פרמטרים ונסביר את הקונפיגורציה שנבחרה.

## 9. אשכולות בשיטת K-Means

בסעיף זה נריץ בהמשך K-Means, נבדוק את ההתאמה בין האשכולות למחלקות ונציג גרפים מתאימים להמחשת המבנה שהתקבל.

## 10. מסווג בייסיאני נאיבי

בסעיף זה נאמן בהמשך שני מסווגים בייסיאניים מסוגים שונים, נציג ביצועים ונבדוק הסתברויות למחלקות עבור רשומת אימות אחת.

## 11. השוואה בין מודלים

בסעיף זה נשווה בהמשך בין המודלים המפוקחים וננסח מסקנה לגבי המודל המתאים ביותר למשימת הסיווג.

## 12. המודל הנבחר

בסעיף זה נציג בהמשך את המודל שנבחר להגשה, את הקונפיגורציה שלו ואת מטריצת הבלבול על סט האימות.

## 13. חיזויים סופיים וייצוא קובץ ההגשה

בסעיף זה נאמן בהמשך את המודל הסופי על נתוני האימון המלאים, נחזה את התוויות עבור קובץ המבחן הסופי ונייצא קובץ אקסל בפורמט קובץ הדוגמה.